In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from bed_reader import open_bed

PLINK_DIR = Path("<PATH_TO_PLINK_ROOT>")
PHENO_TSV = Path("<PATH_TO_BP_WITH_ZERO_TSV>")
DRUG_CLASSES_TSV = Path("<PATH_TO_DRUG_CLASSES_TSV>").expanduser()

RSID = "rs12513069"
CHROM = "4"
BP_POS = 21371225

MEASURE_MIN, MEASURE_MAX = 1, 60

CLASSES_6 = [
    "ACE_inhibitor",
    "angiotensin_receptor_blocker",
    "calcium_channel_blocker",
    "beta_blocker",
    "diuretic",
    "statin",
]

CLASS_DISPLAY = {
    "ACE_inhibitor": "ACEi",
    "angiotensin_receptor_blocker": "ARB",
    "calcium_channel_blocker": "CCB",
    "beta_blocker": "BB",
    "diuretic": "Diuretic",
    "statin": "Statin",
}

CLASS_RECODE_FROM_FILE = {
    "ace_inhibitor": "ACE_inhibitor",
    "angiotensin_ii_receptor_blocker": "angiotensin_receptor_blocker",
    "calcium_channel_blocker": "calcium_channel_blocker",
    "beta_blockers": "beta_blocker",
    "diuretics": "diuretic",
    "statin": "statin",
    "statinm": "statin",
}

def norm_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def display_label(l):
    return CLASS_DISPLAY.get(str(l), str(l))

bar_colors = ["#728A95", "#AEB8AF", "#D2D9D5"]
GENO_ORDER = ["0 minor (ref)", "1 minor (het)", "2 minor (hom)"]
GENO_LABEL = {"0 minor (ref)": "0/0", "1 minor (het)": "0/1", "2 minor (hom)": "1/1"}

def plot_grouped_bars_single(df, metric_col, title, y_max=0.40):
    d = df[df[metric_col].isin(CLASSES_6)].copy()

    # counts + percentages (by genotype)  <-- ADDED
    count_tab = (
        pd.crosstab(d["genotype"], d[metric_col])
        .reindex(index=GENO_ORDER, columns=CLASSES_6)
        .fillna(0)
        .astype(int)
    )
    pct_tab = (count_tab.div(count_tab.sum(axis=1), axis=0) * 100.0).fillna(0.0).round(2)

    table = pd.concat({"n": count_tab, "pct": pct_tab}, axis=1)
    table.to_csv("<PATH_TO_FIGURE_4_B_TABLE_TSV>", sep="\t")
    print("\nLongest therapy class by genotype (counts and % within genotype):")
    print(table.to_string())

    tab = (count_tab.div(count_tab.sum(axis=1), axis=0)).fillna(0.0)
    vals = tab.T.values
    x = np.arange(len(CLASSES_6))
    w = 0.25
    n = d["genotype"].value_counts().reindex(GENO_ORDER).fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(8, 3))
    handles, labels = [], []
    for j, g in enumerate(GENO_ORDER):
        h = ax.bar(x + (j - 1) * w, vals[:, j], width=w, color=bar_colors[j])
        handles.append(h[0])
        labels.append(f"{GENO_LABEL[g]} (n={n[g]})")

    ax.set_ylim(0, y_max)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylabel("Proportion")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([display_label(c) for c in CLASSES_6], rotation=0)
    ax.legend(handles, labels, title="Genotype", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("<PATH_TO_FIGURE_4_B_PNG>", dpi=300, bbox_inches="tight")
    plt.show()

def load_one_variant(plink_dir, rsid, chrom, bp_pos):
    prefixes = [plink_dir / "autosomes", plink_dir / f"c{chrom}"]
    last_err = None
    for pref in prefixes:
        bim = pref.with_suffix(".bim")
        fam = pref.with_suffix(".fam")
        bed = pref.with_suffix(".bed")
        if not (bim.exists() and fam.exists() and bed.exists()):
            continue
        bim_df = pd.read_csv(
            bim, sep=r"\s+", header=None,
            names=["chrom", "snp", "cm", "bp", "a1", "a2"],
            dtype={"chrom": str, "snp": str, "cm": float, "bp": int, "a1": str, "a2": str},
        )
        m = (bim_df["snp"] == rsid) | ((bim_df["chrom"] == str(chrom)) & (bim_df["bp"] == int(bp_pos)))
        if not m.any():
            last_err = f"Variant not found in {pref}"
            continue
        snp_i = int(np.flatnonzero(m.to_numpy())[0])
        row = bim_df.iloc[snp_i]
        fam_df = pd.read_csv(
            fam, sep=r"\s+", header=None,
            names=["fid", "iid", "father", "mother", "sex", "pheno"],
            dtype={"fid": str, "iid": str},
        )
        eids = fam_df["iid"].astype(np.int64).to_numpy()
        bed_obj = open_bed(str(bed), count_A1=True)
        g_a1 = np.asarray(bed_obj.read(index=snp_i)).reshape(-1)
        if g_a1.shape[0] != eids.shape[0]:
            raise ValueError(f"Genotype length ({g_a1.shape[0]}) != FAM length ({eids.shape[0]}) for {pref}")
        g_a1 = pd.Series(g_a1, index=eids, name="a1_count")
        return g_a1, row["a1"], row["a2"], pref
    raise ValueError(last_err or f"No suitable plink prefix found (autosomes/c{chrom})")

g_a1, a1, a2, used_prefix = load_one_variant(PLINK_DIR, RSID, CHROM, BP_POS)

p_a1 = (g_a1.mean(skipna=True) / 2.0)
minor_is_a1 = bool(p_a1 < 0.5)
minor_allele = a1 if minor_is_a1 else a2
minor_dosage = g_a1 if minor_is_a1 else (2.0 - g_a1)

geno = pd.DataFrame({"eid": minor_dosage.index, "minor_dosage": minor_dosage.values}).dropna()
geno["minor_dosage"] = geno["minor_dosage"].round().astype(int)
geno["genotype"] = geno["minor_dosage"].map({0: "0 minor (ref)", 1: "1 minor (het)", 2: "2 minor (hom)"}).astype("category")
geno["genotype"] = geno["genotype"].cat.set_categories(GENO_ORDER, ordered=True)

dc = pd.read_csv(DRUG_CLASSES_TSV, sep="\t", dtype=str)
dc["substance_norm"] = dc["substance"].map(norm_name)
dc["class_norm"] = dc["class"].map(norm_name)
dc["class6"] = dc["class_norm"].map(CLASS_RECODE_FROM_FILE)
substance_to_class6 = dict(zip(dc["substance_norm"], dc["class6"]))

header_cols = pd.read_csv(PHENO_TSV, sep="\t", nrows=0).columns.tolist()
META_COLS = ["eid", "date", "age", "systolic", "diastolic", "GP", "ASCEN"]
drug_cols = [c for c in header_cols if c not in META_COLS]

col_to_class = {c: substance_to_class6.get(norm_name(c), None) for c in drug_cols}
class_to_cols = {cl: [c for c in drug_cols if col_to_class.get(c) == cl] for cl in CLASSES_6}

USECOLS = ["eid", "date"] + [c for cols in class_to_cols.values() for c in cols]
bp = pd.read_csv(PHENO_TSV, sep="\t", usecols=USECOLS, dtype={"eid": "int64"}, low_memory=False)
bp["date"] = pd.to_numeric(bp["date"], errors="coerce").astype("Int64")
bp = bp.dropna(subset=["eid", "date"]).copy()
bp["date"] = bp["date"].astype("int64")

for cl, cols in class_to_cols.items():
    if len(cols) == 0:
        bp[f"active__{cl}"] = False
    else = bp[cols].ge(MEASURE_MIN) & bp[cols].le(MEASURE_MAX)
        bp[f"active__{cl}"] = m.any(axis=1)

active_cols = [f"active__{cl}" for cl in CLASSES_6]
bp["any_active"] = bp[active_cols].any(axis=1)
therapy = bp.loc[bp["any_active"], ["eid", "date"] + active_cols].copy()
treated_eids = pd.Index(therapy["eid"].unique(), name="eid")

span_parts = []
for cl in CLASSES_6:
    col = f"active__{cl}"
    sub = bp.loc[bp[col], ["eid", "date"]]
    if not sub.empty:
        g = sub.groupby("eid")["date"].agg(["min", "max"])
        g[cl] = (g["max"] - g["min"]).astype(np.int64)
        span_parts.append(g[[cl]])
span_df = pd.concat(span_parts, axis=1) if span_parts else pd.DataFrame(index=pd.Index([], name="eid"))
span_df = span_df.reindex(treated_eids).fillna(-1)

max_span = span_df.max(axis=1)
winners_span = span_df.eq(max_span, axis=0)
winners_span = winners_span.loc[max_span >= 0]
long_long = winners_span.reset_index().melt(id_vars="eid", var_name="longest_therapy_class", value_name="is_winner")
long_long = long_long[long_long["is_winner"]][["eid", "longest_therapy_class"]]

gdf = geno.set_index("eid")[["genotype"]].copy()
gdf = gdf.loc[gdf.index.isin(treated_eids)].reset_index()
df_long = gdf.merge(long_long, on="eid", how="inner")

plot_grouped_bars_single(
    df_long,
    "longest_therapy_class",
    f"{RSID} ({CHROM}:{BP_POS}) Ă„â€šĂ‹ÂÄ‚ËĂ˘â‚¬ĹˇĂ‚Â¬Ä‚ËĂ˘â€šÂ¬ÄąÄ„ Longest therapy class by genotype (minor={minor_allele})",
    y_max=0.35,
)
